In [142]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

current_dir = Path.cwd()
project_root = current_dir.parents[2]

classification_models = {
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),
    "random_forest": RandomForestClassifier(random_state=42),
    'extra_trees': ExtraTreesClassifier(random_state=42),
    #"gradient_boosting": GradientBoostingClassifier(),
    "adaboost": AdaBoostClassifier(algorithm="SAMME", random_state=42),
    "logistic_regression": LogisticRegression(random_state=42),
    "sgd": SGDClassifier(loss="log_loss", random_state=42),
    "svc": SVC(),
    "knn": KNeighborsClassifier(),
    "gaussian_nb": GaussianNB()
    
}

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


def evaluate_models_10x10_oof_and_test(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.2,
    random_state: int = 42,
    average: str = "macro",   
    decimals: int = 4,
):
    """
    Evalúa modelos con:
      - Outer: StratifiedShuffleSplit (n veces)
      - Inner: StratifiedKFold para OOF en train
      - Métricas: Accuracy, Precision, Recall, F1, AUC (multiclase OVR)
    AUC:
      - Se calcula SOLO si el estimador soporta predict_proba o decision_function
      - Si no, AUC = np.nan
    """

    y = y_df.iloc[:, 0].to_numpy()
    X = X_df.to_numpy()

    # AUC multiclass requiere scores (n_muestras, n_clases)
    def get_scores_multiclass(fitted_pipe: Pipeline, X_part: np.ndarray):
        """
        Devuelve scores continuos por clase:
          - predict_proba si existe y devuelve (n, n_classes)
          - si no, decision_function si existe y devuelve (n, n_classes)
          - si no, None
        """
        if hasattr(fitted_pipe, "predict_proba"):
            try:
                proba = fitted_pipe.predict_proba(X_part)
                if isinstance(proba, np.ndarray) and proba.ndim == 2:
                    return proba
            except Exception:
                pass

        if hasattr(fitted_pipe, "decision_function"):
            try:
                dec = fitted_pipe.decision_function(X_part)
                if isinstance(dec, np.ndarray) and dec.ndim == 2:
                    return dec
            except Exception:
                pass

        return None

    def auc_multiclass_ovr(y_true: np.ndarray, scores, average: str):
        """
        AUC multiclase con One-vs-Rest.
        Devuelve np.nan si no hay scores válidos o si roc_auc_score falla.
        """
        if scores is None:
            return np.nan
        try:
            return roc_auc_score(y_true, scores, multi_class="ovr", average=average)
        except Exception:
            return np.nan

    def compute_metrics(y_true, y_pred, auc_value=np.nan):
        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, average=average, zero_division=0),
            "Recall": recall_score(y_true, y_pred, average=average, zero_division=0),
            "F1": f1_score(y_true, y_pred, average=average, zero_division=0),
            "AUC": auc_value,
        }

    def summarize(metrics_list, suffix: str):
        df = pd.DataFrame(metrics_list)
        # mean/std ignoran NaN -> ideal para AUC cuando no se pueda calcular
        mean = df.mean(numeric_only=True)
        std = df.std(ddof=1, numeric_only=True)
        return {
            f"{c}_{suffix}": f"{mean[c]:.{decimals}f} ± {std[c]:.{decimals}f}"
            for c in mean.index
        }

    outer = StratifiedShuffleSplit(
        n_splits=outer_splits, test_size=test_size, random_state=random_state
    )

    test_summary_rows = []
    cv_summary_rows = []

    for model_name, estimator in models.items():
        print(f"Evaluating {model_name}...")
        test_metrics_all = []
        cv_metrics_all = []

        for train_idx, test_idx in outer.split(X, y):
            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]

            # ---------- OOF en TRAIN (inner CV) ----------
            inner = StratifiedKFold(
                n_splits=inner_splits, shuffle=True, random_state=random_state
            )

            oof_pred = np.zeros_like(y_train)

            # Para AUC OOF necesitamos scores OOF (n_train, n_classes)
            oof_scores = None

            for tr_idx, val_idx in inner.split(X_train, y_train):
                pipe = Pipeline([
                    ("scaler", StandardScaler()),
                    ("model", clone(estimator)),
                ])
                pipe.fit(X_train[tr_idx], y_train[tr_idx])

                # predicciones OOF (clase)
                oof_pred[val_idx] = pipe.predict(X_train[val_idx])

                # scores OOF (por clase) si se puede
                fold_scores = get_scores_multiclass(pipe, X_train[val_idx])
                if fold_scores is not None:
                    if oof_scores is None:
                        oof_scores = np.full(
                            (len(y_train), fold_scores.shape[1]),
                            np.nan,
                            dtype=float
                        )
                    # asignar scores del fold a sus posiciones val_idx
                    oof_scores[val_idx, :] = fold_scores

            auc_oof = auc_multiclass_ovr(y_train, oof_scores, average=average)
            cv_metrics_all.append(compute_metrics(y_train, oof_pred, auc_value=auc_oof))

            # ---------- TEST ----------
            pipe_full = Pipeline([
                ("scaler", StandardScaler()),
                ("model", clone(estimator)),
            ])
            pipe_full.fit(X_train, y_train)

            test_pred = pipe_full.predict(X_test)

            test_scores = get_scores_multiclass(pipe_full, X_test)
            auc_test = auc_multiclass_ovr(y_test, test_scores, average=average)

            test_metrics_all.append(compute_metrics(y_test, test_pred, auc_value=auc_test))

        test_summary_rows.append(
            pd.Series(summarize(test_metrics_all, "Testing"), name=model_name)
        )
        cv_summary_rows.append(
            pd.Series(summarize(cv_metrics_all, "CV"), name=model_name)
        )

    df_test_summary = pd.DataFrame(test_summary_rows)[
        ["Accuracy_Testing", "Precision_Testing", "Recall_Testing", "F1_Testing", "AUC_Testing"]
    ]
    df_cv_summary = pd.DataFrame(cv_summary_rows)[
        ["Accuracy_CV", "Precision_CV", "Recall_CV", "F1_CV", "AUC_CV"]
    ]

    df_final_summary = pd.concat([df_test_summary, df_cv_summary], axis=1)
    return df_final_summary

# HY3

In [144]:
X_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY3_full.csv", index_col=0)
y_HY3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_full.csv", index_col=0)
print("X_HY3_data shape:", X_HY3_data.shape)
print("y_HY3_data shape:", y_HY3_data.shape)

X_HY3_data shape: (909, 931)
y_HY3_data shape: (909, 1)


In [145]:
df_HY3 = evaluate_models_10x10_oof_and_test(
    X_df=X_HY3_data, 
    y_df=y_HY3_data, 
    models=classification_models, 
    random_state=42
)

df_HY3.head(10)


Evaluating DecisionTreeClassifier...
Evaluating random_forest...
Evaluating extra_trees...
Evaluating adaboost...
Evaluating logistic_regression...
Evaluating sgd...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid value encountered in divide
  prob /= prob.sum(axis=1).reshape((prob.shape[0], -1))
/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/sklearn/linear_model/_base.py:403: RuntimeWarning: invalid valu

Evaluating svc...
Evaluating knn...
Evaluating gaussian_nb...


,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,AUC_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV,AUC_CV
DecisionTreeClassifier,0.8341 ± 0.0230,0.6250 ± 0.0609,0.6170 ± 0.0406,0.6172 ± 0.0448,0.7580 ± 0.0258,0.8253 ± 0.0097,0.6455 ± 0.0383,0.6448 ± 0.0312,0.6440 ± 0.0336,0.7685 ± 0.0168
random_forest,0.9005 ± 0.0131,0.6009 ± 0.0089,0.6183 ± 0.0091,0.6093 ± 0.0090,0.9435 ± 0.0181,0.8986 ± 0.0049,0.7164 ± 0.1118,0.6264 ± 0.0083,0.6262 ± 0.0152,0.9532 ± 0.0088
extra_trees,0.8995 ± 0.0132,0.6004 ± 0.0091,0.6173 ± 0.0093,0.6085 ± 0.0092,0.9460 ± 0.0226,0.8967 ± 0.0049,0.6155 ± 0.0530,0.6168 ± 0.0065,0.6099 ± 0.0104,0.9543 ± 0.0071
adaboost,0.8681 ± 0.0294,0.6820 ± 0.1045,0.6584 ± 0.0520,0.6595 ± 0.0535,0.9065 ± 0.0152,0.8747 ± 0.0077,0.7405 ± 0.0317,0.7176 ± 0.0190,0.7260 ± 0.0194,0.9213 ± 0.0051
logistic_regression,0.8648 ± 0.0257,0.6978 ± 0.1387,0.6497 ± 0.0632,0.6591 ± 0.0782,0.9022 ± 0.0333,0.8711 ± 0.0100,0.7417 ± 0.0391,0.6826 ± 0.0232,0.7024 ± 0.0274,0.8988 ± 0.0174
sgd,0.8599 ± 0.0198,0.7090 ± 0.1573,0.6379 ± 0.0681,0.6502 ± 0.0871,0.8454 ± 0.0282,0.8535 ± 0.0106,0.7381 ± 0.0286,0.6432 ± 0.0265,0.6659 ± 0.0291,0.8425 ± 0.0227
svc,0.8940 ± 0.0206,0.5963 ± 0.0136,0.6136 ± 0.0144,0.6046 ± 0.0140,nan ± nan,0.8878 ± 0.0045,0.5919 ± 0.0030,0.6093 ± 0.0031,0.6004 ± 0.0030,nan ± nan
knn,0.7527 ± 0.0201,0.5269 ± 0.0119,0.5258 ± 0.0132,0.5052 ± 0.0147,0.7672 ± 0.0256,0.7512 ± 0.0046,0.5418 ± 0.0535,0.5267 ± 0.0055,0.5076 ± 0.0095,0.7856 ± 0.0119
gaussian_nb,0.6445 ± 0.0828,0.5991 ± 0.0167,0.6229 ± 0.0737,0.5178 ± 0.0561,0.7881 ± 0.0459,0.6682 ± 0.0863,0.5977 ± 0.0197,0.6087 ± 0.0548,0.5316 ± 0.0574,0.7748 ± 0.0293


# HY4

In [ ]:
X_HY4_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_HY4_full.csv", index_col=0)
y_HY4_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY4_full.csv", index_col=0)
print("X_HY4_data shape:", X_HY4_data.shape)
print("y_HY4_data shape:", y_HY4_data.shape)

X_HY4_data shape: (909, 931)
y_HY4_data shape: (909, 1)


In [ ]:
df_HY4 = evaluate_models_10x10_oof_and_test(
    X_df=X_HY4_data, 
    y_df=y_HY4_data, 
    models=classification_models, 
    random_state=42
)

df_HY4.head(10)

Evaluating DecisionTreeClassifier...
Evaluating random_forest...
Evaluating adaboost...
Evaluating logistic_regression...
Evaluating sgd...
Evaluating svc...
Evaluating knn...
Evaluating gaussian_nb...


,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV
DecisionTreeClassifier,0.7055 ± 0.0242,0.5088 ± 0.0280,0.5067 ± 0.0377,0.5032 ± 0.0304,0.7142 ± 0.0147,0.5067 ± 0.0224,0.5131 ± 0.0254,0.5090 ± 0.0238
random_forest,0.8335 ± 0.0132,0.5307 ± 0.1244,0.4913 ± 0.0131,0.4648 ± 0.0231,0.8304 ± 0.0047,0.5540 ± 0.1243,0.4835 ± 0.0056,0.4528 ± 0.0096
adaboost,0.7775 ± 0.0297,0.6311 ± 0.0813,0.5441 ± 0.0385,0.5568 ± 0.0394,0.7744 ± 0.0097,0.5855 ± 0.0282,0.5494 ± 0.0264,0.5606 ± 0.0270
logistic_regression,0.7571 ± 0.0305,0.5503 ± 0.0758,0.5376 ± 0.0634,0.5367 ± 0.0650,0.7565 ± 0.0120,0.5503 ± 0.0368,0.5273 ± 0.0306,0.5353 ± 0.0324
sgd,0.7780 ± 0.0145,0.6068 ± 0.1162,0.5335 ± 0.0499,0.5477 ± 0.0669,0.7751 ± 0.0094,0.5617 ± 0.0384,0.5231 ± 0.0230,0.5309 ± 0.0289


# MCID

In [ ]:
X_MCID_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_full.csv", index_col=0)
y_MCID_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_full.csv", index_col=0)
print("X_MCID_data shape:", X_MCID_data.shape)
print("y_MCID_data shape:", y_MCID_data.shape)

X_MCID_data shape: (909, 936)
y_MCID_data shape: (909, 1)


In [ ]:
df_MCID = evaluate_models_10x10_oof_and_test(
    X_df=X_MCID_data, 
    y_df=y_MCID_data, 
    models=classification_models, 
    random_state=42
)

df_MCID.head(10)

Evaluating DecisionTreeClassifier...
Evaluating random_forest...
Evaluating adaboost...
Evaluating logistic_regression...
Evaluating sgd...
Evaluating svc...
Evaluating knn...
Evaluating gaussian_nb...


,Accuracy_Testing,Precision_Testing,Recall_Testing,F1_Testing,Accuracy_CV,Precision_CV,Recall_CV,F1_CV
DecisionTreeClassifier,0.4841 ± 0.0346,0.4011 ± 0.0363,0.4034 ± 0.0358,0.4013 ± 0.0360,0.4784 ± 0.0160,0.4002 ± 0.0168,0.3999 ± 0.0170,0.3998 ± 0.0169
random_forest,0.5819 ± 0.0344,0.4621 ± 0.0366,0.4566 ± 0.0289,0.4311 ± 0.0271,0.5773 ± 0.0117,0.4622 ± 0.0202,0.4517 ± 0.0132,0.4285 ± 0.0136
adaboost,0.5604 ± 0.0230,0.4588 ± 0.0473,0.4517 ± 0.0313,0.4403 ± 0.0309,0.5510 ± 0.0087,0.4350 ± 0.0178,0.4366 ± 0.0110,0.4260 ± 0.0128
logistic_regression,0.4995 ± 0.0330,0.4242 ± 0.0299,0.4234 ± 0.0308,0.4226 ± 0.0307,0.4988 ± 0.0187,0.4211 ± 0.0215,0.4208 ± 0.0219,0.4208 ± 0.0217
sgd,0.5143 ± 0.0280,0.4170 ± 0.0277,0.4133 ± 0.0237,0.4108 ± 0.0247,0.5158 ± 0.0186,0.4150 ± 0.0206,0.4128 ± 0.0193,0.4112 ± 0.0202
